"""
Synthetic dataset generator
Creates a relational B2B enterprise model for Revenue Analysis, BI, SQL, ML, Forecastin, PVM
"""

In [5]:
import numpy as np
import pandas as pd 
from pathlib import Path 

In [6]:
def build_dim_date(start="2021-01-01", end="2025-12-31"):
    dates = pd.date_range(start, end, freq="D")
    df = pd.DataFrame({"date": dates})
    df["date_id"] = df["date"].dt.strftime("%Y%m%d").astype(int)
    df["year"] = df["date"].dt.year
    df["quarter"] = df["date"].dt.quarter
    df["month"] = df["date"].dt.month
    df["month_name"] = df["date"].dt.month_name()
    df["year_month"] = df["date"].dt.to_period("M").astype(str)
    df["day_of_week"] = df["date"].dt.dayofweek + 1
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["is_month_end"] = df["date"].dt.is_month_end
    df["is_quarter_end"] = df["date"].dt.is_quarter_end
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df  

In [20]:
def generate_dataset(output_dir="../data/generated_data_table", n_sales_rows = 120_000, seed=42):
    rng = np.random.default_rng(seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    dim_date = build_dim_date()
    date_series = pd.to_datetime(dim_date["date"])
    date_array = date_series.to_numpy()

    dim_region = pd.DataFrame([
        (1, "Germany", "DACH", "EUR", 1.00, 1.06, 0.98),
        (2, "France", "Western Europe", "EUR", 1.00, 1.02, 0.97),
        (3, "Spain", "Southern Europe", "EUR", 1.00, 1.09, 0.94),
        (4, "Italy", "Southern Europe", "EUR", 1.00, 1.01, 0.93),
        (5, "United Kingdom", "Northern Europe", "GBP", 1.17, 0.97, 1.01),
        (6, "United States", "North America", "USD", 0.92, 1.14, 1.03),
        (7, "Canada", "North America", "CAD", 0.68, 1.04, 0.99),
        (8, "Netherlands", "Benelux", "EUR", 1.00, 1.07, 1.00),
        (9, "Poland", "Eastern Europe", "PLN", 0.23, 1.12, 0.89),
        (10, "Japan", "APAC", "JPY", 0.0062, 0.95, 1.02),
    ], columns=["region_id", "country", "region", "currency", "fx_to_eur", "market_growth_factor", "margin_factor"])

    product_groups = [
        ("Laptop Pro", "IT Devices", 950, 0.29, 1.08),
        ("Laptop Standard", "IT Devices", 620, 0.24, 1.01),
        ("Tablet Enterprise", "IT Devices", 410, 0.27, 1.12),
        ("Medical Sensor", "Medical Devices", 180, 0.45, 1.15),
        ("Monitoring Device", "Medical Devices", 760, 0.39, 1.10),
        ("Industrial Scanner", "Industrial Tech", 1450, 0.34, 1.03),
        ("Connectivity Module", "Components", 85, 0.31, 1.18),
        ("Legacy Workstation", "Legacy Products", 780, 0.18, 0.88),
        ("Accessory Kit", "Accessories", 55, 0.22, 1.05),
        ("Service Contract", "Services", 240, 0.62, 1.16),]
    
    product_rows, product_id = [], 1000
    for grp, family, base_price, base_margin, growth_factor in product_groups:
        for i in range(1, 21):
            lifecycle = rng.choice(["Growth", "Mature", "Decline", "New"], p=[.28, .45, .17, .10])
            if "Legacy" in grp:
                lifecycle = rng.choice(["Mature", "Decline"], p=[.25, .75])
            price = base_price * rng.lognormal(0, 0.16)
            unit_cost = price * (1 - np.clip(base_margin + rng.normal(0, 0.05), 0.08, 0.75))
            product_rows.append([
                product_id, f"SKU-{product_id}", family, grp, f"{grp} Model {i:02d}",
                int(rng.choice([2018, 2019, 2020, 2021, 2022, 2023, 2024], p=[.08,.10,.15,.22,.18,.17,.10])),
                lifecycle, round(price, 2), round(unit_cost, 2), round(base_margin, 3), growth_factor
            ])
            product_id += 1

    dim_product = pd.DataFrame(product_rows, columns=[
        "product_id", "sku", "product_family", "product_group", "product_name",
        "launch_year", "lifecycle_stage", "base_list_price_eur", "base_unit_cost_eur",
        "target_margin_pct", "product_growth_factor"
    ])

    customer_segments = ["Enterprise", "Mid-Market", "SMB", "Public Sector", "Distributor"]
    segment_scale = {"Enterprise": 8.0, "Mid-Market": 3.5, "SMB": 1.0, "Public Sector": 2.4, "Distributor": 5.5}
    churn_risk = {"Enterprise": .03, "Mid-Market": .07, "SMB": .16, "Public Sector": .05, "Distributor": .09}
    customer_rows = []

    for cid in range(1, 1201):
        segment = rng.choice(customer_segments, p=[.12, .28, .38, .10, .12])
        region_id = int(rng.choice(dim_region["region_id"], p=np.array([.18,.11,.10,.09,.10,.16,.06,.08,.07,.05])))
        start_date = pd.Timestamp("2020-01-01") + pd.Timedelta(days=int(rng.integers(0, 1500)))
        industry = rng.choice(["Healthcare", "Manufacturing", "Retail", "Technology", "Logistics", "Education", "Government"])
        size_score = rng.lognormal(mean=np.log(segment_scale[segment]), sigma=0.55)
        customer_rows.append([
            cid, f"CUST-{cid:05d}", segment, industry, region_id,
            start_date.strftime("%Y-%m-%d"), round(size_score, 3),
            round(churn_risk[segment] * rng.uniform(.65, 1.45), 3)
        ])    

    dim_customer = pd.DataFrame(customer_rows, columns=[
        "customer_id", "customer_code", "customer_segment", "industry", "region_id",
        "customer_since", "customer_size_score", "base_churn_probability"
    ])

    rep_rows = []
    for rid in range(1, 86):
        region_id = int(rng.choice(dim_region["region_id"]))
        seniority = rng.choice(["Junior", "Professional", "Senior", "Key Account"], p=[.18,.42,.28,.12])
        quota = {"Junior": 750_000, "Professional": 1_400_000, "Senior": 2_300_000, "Key Account": 4_000_000}[seniority]
        rep_rows.append([rid, f"REP-{rid:03d}", region_id, seniority, int(quota * rng.uniform(.8, 1.25))])

    dim_sales_rep = pd.DataFrame(rep_rows, columns=["sales_rep_id", "sales_rep_code", "region_id", "seniority", "annual_quota_eur"])

    month = date_series.dt.month.to_numpy()
    year = date_series.dt.year.to_numpy()
    season = np.where(np.isin(month, [10, 11, 12]), 1.30, np.where(np.isin(month, [7, 8]), 0.82, 1.00))
    year_growth = 1.06 ** (year - 2021)
    date_weights = season * year_growth
    date_weights = date_weights / date_weights.sum()

    product_weights = dim_product["product_growth_factor"].to_numpy() * np.where(
        dim_product["lifecycle_stage"].eq("Decline"), 0.55,
        np.where(dim_product["lifecycle_stage"].eq("New"), 0.65, 1.0)
    )
    product_weights = product_weights / product_weights.sum()

    customer_weights = dim_customer["customer_size_score"].to_numpy()
    customer_weights = customer_weights / customer_weights.sum()

    sales_dates = rng.choice(date_array, size=n_sales_rows, p=date_weights)
    fact_sales = pd.DataFrame({
        "sales_id": np.arange(1, n_sales_rows + 1),
        "date": pd.to_datetime(sales_dates),
        "customer_id": rng.choice(dim_customer["customer_id"], size=n_sales_rows, p=customer_weights),
        "product_id": rng.choice(dim_product["product_id"], size=n_sales_rows, p=product_weights)
    })

    fact_sales = fact_sales.merge(dim_customer[["customer_id", "region_id", "customer_segment", "customer_size_score", "base_churn_probability"]], on="customer_id", how="left")
    fact_sales = fact_sales.merge(dim_product[["product_id", "product_group", "product_family", "base_list_price_eur", "base_unit_cost_eur", "lifecycle_stage", "launch_year", "product_growth_factor"]], on="product_id", how="left")
    fact_sales = fact_sales.merge(dim_region[["region_id", "currency", "fx_to_eur", "market_growth_factor", "margin_factor"]], on="region_id", how="left")

    transaction_year = fact_sales["date"].dt.year.to_numpy()
    inflation = 1 + 0.035 * (transaction_year - 2021)
    lifecycle_price_factor = np.select(
        [fact_sales["lifecycle_stage"].eq("New"), fact_sales["lifecycle_stage"].eq("Growth"), fact_sales["lifecycle_stage"].eq("Decline")],
        [1.12, 1.05, 0.93],
        default=1.0
    )

    discount = np.select(
        [fact_sales["customer_segment"].eq("Enterprise"), fact_sales["customer_segment"].eq("Distributor"), fact_sales["customer_segment"].eq("SMB")],
        [rng.normal(.14, .04, n_sales_rows), rng.normal(.20, .05, n_sales_rows), rng.normal(.05, .03, n_sales_rows)],
        default=rng.normal(.09, .04, n_sales_rows)
    )
    discount = np.clip(discount, 0, .38)

    units_base = np.select(
        [fact_sales["customer_segment"].eq("Enterprise"), fact_sales["customer_segment"].eq("Distributor"), fact_sales["customer_segment"].eq("Mid-Market"), fact_sales["customer_segment"].eq("SMB")],
        [rng.poisson(45, n_sales_rows), rng.poisson(70, n_sales_rows), rng.poisson(18, n_sales_rows), rng.poisson(6, n_sales_rows)],
        default=rng.poisson(12, n_sales_rows)
    )
    units = np.maximum(1, (units_base * rng.lognormal(0, .35, n_sales_rows)).astype(int))
    outlier_mask = rng.random(n_sales_rows) < 0.012
    units[outlier_mask] *= rng.integers(4, 12, outlier_mask.sum())

    asp_eur = fact_sales["base_list_price_eur"].to_numpy() * inflation * lifecycle_price_factor * (1 - discount) * rng.normal(1, .04, n_sales_rows)
    asp_eur = np.maximum(5, asp_eur)
    revenue_eur = units * asp_eur
    unit_cost_eur = fact_sales["base_unit_cost_eur"].to_numpy() * (1 + 0.025 * (transaction_year - 2021)) * rng.normal(1, .035, n_sales_rows)
    gross_profit_eur = revenue_eur - units * unit_cost_eur
    margin_pct = gross_profit_eur / revenue_eur

    fact_sales["date_id"] = fact_sales["date"].dt.strftime("%Y%m%d").astype(int)
    fact_sales["year_month"] = fact_sales["date"].dt.to_period("M").astype(str)
    fact_sales["sales_rep_id"] = rng.choice(dim_sales_rep["sales_rep_id"], size=n_sales_rows)
    fact_sales["order_id"] = "ORD-" + fact_sales["sales_id"].astype(str).str.zfill(8)
    fact_sales["units"] = units
    fact_sales["asp_eur"] = np.round(asp_eur, 2)
    fact_sales["discount_pct"] = np.round(discount, 3)
    fact_sales["revenue_eur"] = np.round(revenue_eur, 2)
    fact_sales["revenue_local_currency"] = np.round(fact_sales["revenue_eur"] / fact_sales["fx_to_eur"], 2)
    fact_sales["unit_cost_eur"] = np.round(unit_cost_eur, 2)
    fact_sales["gross_profit_eur"] = np.round(gross_profit_eur, 2)
    fact_sales["gross_margin_pct"] = np.round(margin_pct, 4)
    fact_sales["is_outlier_order"] = outlier_mask

    for col, rate in {"discount_pct": .008, "sales_rep_id": .006, "gross_margin_pct": .004}.items():
        fact_sales.loc[rng.random(n_sales_rows) < rate, col] = np.nan

    fact_sales["date"] = fact_sales["date"].dt.strftime("%Y-%m-%d")
    fact_sales = fact_sales[[
        "sales_id", "order_id", "date_id", "date", "year_month", "customer_id", "product_id",
        "region_id", "sales_rep_id", "units", "asp_eur", "discount_pct", "revenue_eur",
        "revenue_local_currency", "currency", "unit_cost_eur", "gross_profit_eur",
        "gross_margin_pct", "is_outlier_order"
    ]]

    monthly = fact_sales.groupby(["year_month", "product_id", "region_id"], dropna=False).agg(
        actual_units=("units", "sum"),
        actual_revenue_eur=("revenue_eur", "sum"),
        actual_gp_eur=("gross_profit_eur", "sum")
    ).reset_index()

    region_bias = dim_region.set_index("region_id")["market_growth_factor"].to_dict()
    monthly["forecast_bias_factor"] = monthly["region_id"].map(region_bias).fillna(1.0)
    monthly["forecast_units"] = np.maximum(0, (monthly["actual_units"] * (1 + rng.normal(0, 0.13, len(monthly))) / monthly["forecast_bias_factor"]).round().astype(int))
    monthly["forecast_revenue_eur"] = np.round(monthly["actual_revenue_eur"] * (1 + rng.normal(0, .15, len(monthly))), 2)
    monthly["forecast_version"] = rng.choice(["Budget", "Rolling Forecast", "Latest Estimate"], size=len(monthly), p=[.25,.50,.25])
    fact_forecast = monthly[["year_month", "product_id", "region_id", "forecast_version", "forecast_units", "forecast_revenue_eur", "actual_units", "actual_revenue_eur"]].copy()
    fact_forecast.insert(0, "forecast_id", np.arange(1, len(fact_forecast) + 1))

    fact_inventory = monthly[["year_month", "product_id", "region_id", "actual_units"]].copy()
    fact_inventory["opening_stock_units"] = np.maximum(0, (fact_inventory["actual_units"] * rng.uniform(.6, 1.8, len(fact_inventory))).astype(int))
    fact_inventory["production_units"] = np.maximum(0, (fact_inventory["actual_units"] * rng.uniform(.75, 1.45, len(fact_inventory))).astype(int))
    fact_inventory["ending_stock_units"] = np.maximum(0, fact_inventory["opening_stock_units"] + fact_inventory["production_units"] - fact_inventory["actual_units"])
    fact_inventory["stockout_flag"] = fact_inventory["ending_stock_units"] < fact_inventory["actual_units"] * .12
    fact_inventory["inventory_value_eur"] = np.round(fact_inventory["ending_stock_units"] * rng.uniform(35, 650, len(fact_inventory)), 2)
    fact_inventory.insert(0, "inventory_id", np.arange(1, len(fact_inventory) + 1))
    fact_inventory = fact_inventory.drop(columns=["actual_units"])

    months = pd.period_range("2021-01", "2025-12", freq="M").astype(str)
    cost_rows, cost_id = [], 1
    for ym in months:
        y = int(ym[:4])
        for _, row in dim_product[["product_id", "base_unit_cost_eur"]].iterrows():
            std = row["base_unit_cost_eur"] * (1 + .025 * (y - 2021)) * rng.normal(1, .025)
            cost_rows.append([cost_id, ym, int(row["product_id"]), round(std, 2), round(std * rng.uniform(.95, 1.08), 2)])
            cost_id += 1
    fact_costs = pd.DataFrame(cost_rows, columns=["cost_id", "year_month", "product_id", "standard_unit_cost_eur", "actual_unit_cost_eur"])

    n_activities = 35_000
    crm_dates = pd.to_datetime(rng.choice(date_array, n_activities))
    fact_crm = pd.DataFrame({
        "activity_id": np.arange(1, n_activities + 1),
        "date": crm_dates,
        "customer_id": rng.choice(dim_customer["customer_id"], n_activities, p=customer_weights),
        "sales_rep_id": rng.choice(dim_sales_rep["sales_rep_id"], n_activities),
        "activity_type": rng.choice(["Call", "Email", "Demo", "Visit", "Business Review", "Training"], n_activities, p=[.34,.32,.10,.12,.06,.06]),
        "activity_minutes": np.maximum(5, rng.normal(35, 20, n_activities).astype(int)),
        "sentiment_score": np.round(rng.normal(.08, .55, n_activities), 3)
    })
    fact_crm["date_id"] = fact_crm["date"].dt.strftime("%Y%m%d").astype(int)
    fact_crm["customer_health_score"] = np.round(np.clip(65 + fact_crm["sentiment_score"] * 15 + rng.normal(0, 8, n_activities), 1, 100), 1)
    fact_crm["date"] = fact_crm["date"].dt.strftime("%Y-%m-%d")
    fact_crm = fact_crm[["activity_id", "date_id", "date", "customer_id", "sales_rep_id", "activity_type", "activity_minutes", "sentiment_score", "customer_health_score"]]

    sampled_sales = fact_sales.sample(frac=.035, random_state=seed)
    fact_returns = sampled_sales[["sales_id", "date_id", "date", "customer_id", "product_id", "region_id", "units", "revenue_eur"]].copy()
    fact_returns.insert(0, "return_id", np.arange(1, len(fact_returns) + 1))
    fact_returns["return_units"] = np.maximum(1, (fact_returns["units"] * rng.uniform(.05, .5, len(fact_returns))).astype(int))
    fact_returns["return_value_eur"] = np.round(fact_returns["revenue_eur"] * fact_returns["return_units"] / fact_returns["units"], 2)
    fact_returns["return_reason"] = rng.choice(["Defect", "Wrong Configuration", "Customer Cancellation", "Shipping Damage", "Other"], len(fact_returns), p=[.35,.22,.18,.15,.10])
    fact_returns = fact_returns.drop(columns=["units", "revenue_eur"])

    n_opp = 15_000
    opp_dates = pd.to_datetime(rng.choice(date_array, n_opp))
    fact_pipeline = pd.DataFrame({
        "opportunity_id": np.arange(1, n_opp + 1),
        "created_date": opp_dates,
        "customer_id": rng.choice(dim_customer["customer_id"], n_opp, p=customer_weights),
        "product_group": rng.choice([p[0] for p in product_groups], n_opp),
        "sales_rep_id": rng.choice(dim_sales_rep["sales_rep_id"], n_opp),
        "stage": rng.choice(["Lead", "Qualified", "Proposal", "Negotiation", "Closed Won", "Closed Lost"], n_opp, p=[.25,.22,.18,.13,.12,.10]),
        "expected_value_eur": np.round(rng.lognormal(10.2, 1.0, n_opp), 2),
        "win_probability": np.round(rng.beta(2.2, 3.0, n_opp), 3)
    })
    fact_pipeline["expected_close_date"] = (fact_pipeline["created_date"] + pd.to_timedelta(rng.integers(20, 220, n_opp), unit="D")).dt.strftime("%Y-%m-%d")
    fact_pipeline["created_date_id"] = fact_pipeline["created_date"].dt.strftime("%Y%m%d").astype(int)
    fact_pipeline["created_date"] = fact_pipeline["created_date"].dt.strftime("%Y-%m-%d")
    fact_pipeline["weighted_pipeline_eur"] = np.round(fact_pipeline["expected_value_eur"] * fact_pipeline["win_probability"], 2)

    tables = {
        "dimDate": dim_date,
        "dimRegion": dim_region,
        "dimProduct": dim_product,
        "dimCustomer": dim_customer,
        "dimSalesRep": dim_sales_rep,
        "factSales": fact_sales,
        "factForecast": fact_forecast,
        "factInventory": fact_inventory,
        "factCosts": fact_costs,
        "factCRMActivities": fact_crm,
        "factReturns": fact_returns,
        "factPipeline": fact_pipeline,
    }

    for name, df in tables.items():
        df.to_csv(output_dir / f"{name}.csv", sep=";", decimal=",", index=False)

    return tables


In [21]:
if __name__ == "__main__":
    tables = generate_dataset(output_dir="../data/generated_data_tables", n_sales_rows=120_000, seed=42)
    print("Generated tables:")
    for name, df in tables.items():
        print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

Generated tables:
dimDate: 1,826 rows x 11 columns
dimRegion: 10 rows x 7 columns
dimProduct: 200 rows x 11 columns
dimCustomer: 1,200 rows x 8 columns
dimSalesRep: 85 rows x 5 columns
factSales: 120,000 rows x 19 columns
factForecast: 69,852 rows x 9 columns
factInventory: 69,852 rows x 9 columns
factCosts: 12,000 rows x 5 columns
factCRMActivities: 35,000 rows x 9 columns
factReturns: 4,200 rows x 10 columns
factPipeline: 15,000 rows x 11 columns
